# Recursive NSPPK Feature Generation on HIV

This experiment compares named, unfiltered NSPPK baselines with a recursive pipeline. Each stage fits `ImportanceNSPPK` on the training split, ranks each graph's current candidate roots, and restricts the next stage to those nodes. The classifier can use each individual stage's vector or the concatenated vectors from all stages.

The default stages start with radius 0–1 and distance 0–4 without connectors, expand to radius 0–2 with connector 1, then add simplex fragments. `BASELINES` is a list of named full-feature configurations and defaults to unfiltered NSPPK at each stage's settings. The experiment uses a fixed, balanced sample and held-out split; ROC AUC is the primary metric.


In [ ]:
from pathlib import Path
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import hstack
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(path for path in REPO_CANDIDATES if (path / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsppk import NSPPK
from recursive_nsppk import RecursiveNSPPKClassifier

RANDOM_STATE = 2026
NBITS = 12
SIMPLEX_MAX_ORDER = 2
SAMPLE_PER_CLASS = 100
ROOT_RETENTION = 0.25  # Positive integer = top-k nodes; fraction in (0, 1) = per-graph share.
N_ITER = 3
N_ESTIMATORS = 100
HIV_PICKLE = REPO_ROOT / 'data' / 'HIV_networkx_graphs.pkl'


In [ ]:
with HIV_PICKLE.open('rb') as handle:
    hiv_data = pickle.load(handle)
hiv_graphs_all = hiv_data['graphs']
hiv_labels_all = np.asarray(hiv_data['labels'], dtype=int)

rng = np.random.default_rng(RANDOM_STATE)
class_indices = [np.flatnonzero(hiv_labels_all == label) for label in (0, 1)]
if any(len(indices) < SAMPLE_PER_CLASS for indices in class_indices):
    raise ValueError('SAMPLE_PER_CLASS exceeds the available HIV examples in a class.')
sampled_indices = np.concatenate([
    rng.choice(indices, size=SAMPLE_PER_CLASS, replace=False)
    for indices in class_indices
])
rng.shuffle(sampled_indices)
hiv_graphs = [hiv_graphs_all[index] for index in sampled_indices]
hiv_labels = hiv_labels_all[sampled_indices]
train_indices, test_indices = train_test_split(
    np.arange(len(hiv_graphs)), test_size=0.25,
    random_state=RANDOM_STATE, stratify=hiv_labels,
)
train_graphs = [hiv_graphs[index] for index in train_indices]
y_train = hiv_labels[train_indices]
print(f'Full bundled dataset: {len(hiv_graphs_all):,} graphs; class counts={np.bincount(hiv_labels_all).tolist()}')
print(f'Benchmark sample: {len(hiv_graphs):,} balanced graphs; train={len(train_indices)}, test={len(test_indices)}')


## Stage recipe, baselines, and root selection

`STAGES` is a list of dictionaries, one per recursive pass. `BASELINES` is a list of named dictionaries that can be edited to add, remove, or change full-feature comparisons. By default it includes the unfiltered counterpart of every stage. Change `ROOT_RETENTION` to an integer count or a fraction between zero and one.


In [ ]:
STAGES = [
    {'radius': 1, 'distance': 4, 'connector': 0, 'use_simplex_features': False},
    {'radius': 2, 'distance': 4, 'connector': 1, 'use_simplex_features': False},
    {'radius': 2, 'distance': 4, 'connector': 1, 'use_simplex_features': True},
]
BASELINES = [
    {
        'name': 'All roots: stage 1 settings',
        'params': {'radius': 1, 'distance': 4, 'connector': 0, 'use_simplex_features': False},
    },
    {
        'name': 'All roots: stage 2 settings',
        'params': {'radius': 2, 'distance': 4, 'connector': 1, 'use_simplex_features': False},
    },
    {
        'name': 'All roots: stage 3 settings',
        'params': {'radius': 2, 'distance': 4, 'connector': 1, 'use_simplex_features': True},
    },
]


def fit_classifier(feature_matrix):
    classifier = LogisticRegression(
        solver='liblinear', max_iter=500, class_weight='balanced',
        random_state=RANDOM_STATE,
    )
    classifier.fit(feature_matrix[train_indices], y_train)
    return classifier


## Unfiltered NSPPK baselines

Each named baseline is encoded with all graph nodes as roots. Feature generation, classifier fitting, and prediction are timed separately.


In [ ]:
baseline_rows = []
baseline_features_by_name = {}
for baseline in BASELINES:
    encoder = NSPPK(
        **baseline['params'], nbits=NBITS, dense=False, parallel=False,
        simplex_feature_max_order=SIMPLEX_MAX_ORDER,
    )
    started = time.perf_counter()
    features = encoder.fit(train_graphs, y_train).transform(hiv_graphs)
    feature_seconds = time.perf_counter() - started

    started = time.perf_counter()
    classifier = fit_classifier(features)
    fit_seconds = time.perf_counter() - started
    started = time.perf_counter()
    probabilities = classifier.predict_proba(features[test_indices])[:, 1]
    predict_seconds = time.perf_counter() - started
    auc = roc_auc_score(hiv_labels[test_indices], probabilities)
    baseline_features_by_name[baseline['name']] = features
    baseline_rows.append({
        'configuration': baseline['name'],
        'ROC AUC': auc,
        'feature_generation_sec': feature_seconds,
        'importance_selection_sec': 0.0,
        'classifier_fit_sec': fit_seconds,
        'prediction_sec': predict_seconds,
        'end_to_end_sec': feature_seconds + fit_seconds + predict_seconds,
        'feature_nnz': int(features.nnz),
    })
    print(f"{baseline['name']}: feature generation={feature_seconds:.2f}s, ROC AUC={auc:.4f}")


## Recursive feature generation through the classifier API

The classifier learns the staged selectors on the training split. `transform_stages` returns each individual stage's graph vectors for stage-only AUC diagnostics; `predict_proba` evaluates the fitted cumulative classifier.


In [ ]:
recursive_classifier = RecursiveNSPPKClassifier(
    estimator=LogisticRegression(
        solver='liblinear', max_iter=500, class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    stages=STAGES,
    root_retention=ROOT_RETENTION,
    nbits=NBITS,
    dense=False,
    parallel=False,
    simplex_feature_max_order=SIMPLEX_MAX_ORDER,
    importance_n_iter=N_ITER,
    importance_n_estimators=N_ESTIMATORS,
    importance_quantile=0.5,
)

fit_started = time.perf_counter()
recursive_classifier.fit(train_graphs, y_train)
recursive_fit_seconds = time.perf_counter() - fit_started

transform_started = time.perf_counter()
stage_feature_matrices = recursive_classifier.transform_stages(hiv_graphs)
recursive_transform_seconds = time.perf_counter() - transform_started
recursive_features = hstack(stage_feature_matrices, format='csr')

stage_rows = []
for stage_index, (stage_params, stage_features, timing, candidates, selected) in enumerate(zip(
    recursive_classifier.stage_params_,
    recursive_classifier.stage_feature_matrices_,
    recursive_classifier.stage_timings_,
    recursive_classifier.stage_input_roots_,
    recursive_classifier.stage_selected_roots_,
), start=1):
    stage_rows.append({
        'stage': stage_index,
        **stage_params,
        'mean_input_roots': float(np.mean([len(roots) for roots in candidates])),
        'mean_selected_roots': float(np.mean([len(roots) for roots in selected])),
        **timing,
        'training_feature_nnz': int(stage_features.nnz),
    })

stage_results_df = pd.DataFrame(stage_rows)
display(stage_results_df.round(4))
print(
    f'Recursive classifier fit={recursive_fit_seconds:.2f}s; '
    f'transform all sample stages={recursive_transform_seconds:.2f}s'
)


## Results by baseline and recursive stage

Each individual recursive stage is scored with the same classifier type used by the cumulative model. The final estimator reports cumulative held-out ROC AUC. Runtime results compare the named unfiltered baselines with the complete recursive fit and prediction path; stage-level selector and feature-generation times appear above.


In [ ]:
stage_auc_rows = []
for stage_index, stage_features in enumerate(stage_feature_matrices, start=1):
    stage_classifier = clone(recursive_classifier.estimator_)
    stage_classifier.fit(stage_features[train_indices], y_train)
    stage_probabilities = stage_classifier.predict_proba(stage_features[test_indices])[:, 1]
    stage_auc_rows.append({
        'configuration': f'Recursive stage {stage_index} only',
        'ROC AUC': roc_auc_score(hiv_labels[test_indices], stage_probabilities),
        'feature_nnz': int(stage_features.nnz),
    })

predict_started = time.perf_counter()
recursive_probabilities = recursive_classifier.predict_proba(
    [hiv_graphs[index] for index in test_indices]
)[:, 1]
recursive_predict_seconds = time.perf_counter() - predict_started
recursive_auc = roc_auc_score(hiv_labels[test_indices], recursive_probabilities)

auc_rows = [
    {'configuration': row['configuration'], 'ROC AUC': row['ROC AUC'], 'feature_nnz': row['feature_nnz']}
    for row in baseline_rows
]
auc_rows.extend(stage_auc_rows)
auc_rows.append({
    'configuration': 'Recursive cumulative',
    'ROC AUC': recursive_auc,
    'feature_nnz': int(recursive_features.nnz),
})
auc_results_df = pd.DataFrame(auc_rows)
display(auc_results_df.round(4))

runtime_rows = [
    {
        'configuration': row['configuration'],
        'fit_sec': row['feature_generation_sec'] + row['classifier_fit_sec'],
        'prediction_sec': row['prediction_sec'],
        'end_to_end_sec': row['end_to_end_sec'],
    }
    for row in baseline_rows
]
runtime_rows.append({
    'configuration': 'Recursive cumulative',
    'fit_sec': recursive_fit_seconds,
    'prediction_sec': recursive_predict_seconds,
    'end_to_end_sec': recursive_fit_seconds + recursive_predict_seconds,
})
runtime_results_df = pd.DataFrame(runtime_rows)
display(runtime_results_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
colors = ['#4C78A8' if name.startswith('All roots:') else '#F58518' for name in auc_results_df['configuration']]
axes[0].barh(auc_results_df['configuration'], auc_results_df['ROC AUC'], color=colors)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel('ROC AUC')
axes[0].set_title('Held-out ROC AUC')
axes[1].barh(runtime_results_df['configuration'], runtime_results_df['end_to_end_sec'], color=['#4C78A8'] * len(baseline_rows) + ['#F58518'])
axes[1].set_xlabel('Seconds')
axes[1].set_title('End-to-end runtime')
for ax in axes:
    ax.grid(axis='x', alpha=0.25)


## Reading the results

- `BASELINES` controls the named unfiltered NSPPK configurations. Add or remove dictionaries to change the comparison set.
- Individual recursive-stage AUCs use one stage's vector alone. The cumulative classifier concatenates all stages, while each later stage generates features only from nodes selected earlier.
- Importance selection is trained on the training split only. Root ranking is per graph; ties retain graph node order.
- This is one fixed balanced sample and split. Treat the AUC and timings as exploratory; repeat with multiple seeds for more stable estimates.
